In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4050 Laptop GPU


### Hyperparameters

In [3]:
batch_size = 64       # how many sequences train on gpu in parallel
block_size = 128      # Context Length
max_iters  = 3000 
eval_intervals = 300  # checking validation loss
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 100
n_embd = 192          # embedding dimension
n_head = 6            # number of Attention heads
n_layer = 6           # number of transformer blocks
dropout = 0.2
torch.manual_seed(42)

### Loading Data

In [4]:
with open('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

In [5]:
text

"A Field Guide to Machine Learning and Artificial Intelligence\n\nIntroduction\n\nArtificial intelligence is the broad effort to build machines that perform tasks which, if done by a person, would be said to require intelligence. Machine learning is the dominant approach to building such machines today. Instead of writing explicit rules for every situation, an engineer collects examples of the behavior they want and lets an algorithm find the pattern that connects inputs to outputs. The algorithm then applies that pattern to new, unseen inputs. This shift from hand written rules to learned patterns is the central idea of the modern field.\n\nEarly History\n\nThe dream of thinking machines predates computers themselves. Philosophers speculated about mechanical reasoning for centuries, but the practical study of artificial intelligence began in the middle of the twentieth century, once electronic computers made it possible to test ideas about reasoning and learning. Early researchers bui

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
vocab_size

57

### Character Tokenizer

In [7]:
stoi = { ch:i for i, ch in enumerate(chars)} # string to integer
itos = { i:ch for i, ch in enumerate(chars)} # integer to string

In [8]:
print(f'stoi : {stoi}')
print(f'itos : {itos}')

stoi : {'\n': 0, ' ': 1, "'": 2, ',': 3, '.': 4, ':': 5, ';': 6, 'A': 7, 'B': 8, 'C': 9, 'D': 10, 'E': 11, 'F': 12, 'G': 13, 'H': 14, 'I': 15, 'J': 16, 'K': 17, 'L': 18, 'M': 19, 'N': 20, 'O': 21, 'P': 22, 'Q': 23, 'R': 24, 'S': 25, 'T': 26, 'U': 27, 'V': 28, 'W': 29, 'Y': 30, 'a': 31, 'b': 32, 'c': 33, 'd': 34, 'e': 35, 'f': 36, 'g': 37, 'h': 38, 'i': 39, 'j': 40, 'k': 41, 'l': 42, 'm': 43, 'n': 44, 'o': 45, 'p': 46, 'q': 47, 'r': 48, 's': 49, 't': 50, 'u': 51, 'v': 52, 'w': 53, 'x': 54, 'y': 55, 'z': 56}
itos : {0: '\n', 1: ' ', 2: "'", 3: ',', 4: '.', 5: ':', 6: ';', 7: 'A', 8: 'B', 9: 'C', 10: 'D', 11: 'E', 12: 'F', 13: 'G', 14: 'H', 15: 'I', 16: 'J', 17: 'K', 18: 'L', 19: 'M', 20: 'N', 21: 'O', 22: 'P', 23: 'Q', 24: 'R', 25: 'S', 26: 'T', 27: 'U', 28: 'V', 29: 'W', 30: 'Y', 31: 'a', 32: 'b', 33: 'c', 34: 'd', 35: 'e', 36: 'f', 37: 'g', 38: 'h', 39: 'i', 40: 'j', 41: 'k', 42: 'l', 43: 'm', 44: 'n', 45: 'o', 46: 'p', 47: 'q', 48: 'r', 49: 's', 50: 't', 51: 'u', 52: 'v', 53: 'w', 54:

In [9]:
encode = lambda x : [stoi[c] for c in x]
encode('Hello')

[14, 35, 42, 42, 45]

In [10]:
decode = lambda x : ''.join([itos[c] for c in x])
decode([14, 35, 42, 42, 45])

'Hello'

In [11]:
# Convert to Tensor
data = torch.tensor(encode(text), dtype=torch.long)
data

tensor([ 7,  1, 12,  ..., 44, 49,  4])

### Train / Validation Split

In [12]:
n = int(0.9*len(data)) # 90% of Data
train_data = data[:n]
val_data = data[n:]

In [13]:
print(f"Train Data : {len(train_data)}")
print(f"val Data : {len(val_data)}")

Train Data : 15287
val Data : 1699


In [14]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size,(batch_size,))
    x = torch.stack([d[i:i+batch_size] for i in ix])
    y = torch.stack([d[i+1:i+batch_size+1] for i in ix])
    return x.to(device), y.to(device)

In [15]:
@torch.no_grad() # no computation graph
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

###  3. Self-attention head

In [16]:
class Head(nn.Module):
    def __init__(self, head_size):
        super.__init__()
        self.key = nn.Linear(n_embd, head_size, bais=False)
        self.query  = nn.Linear(n_embd, head_size, bais=False)
        self.value  = nn.Linear(n_embd, head_size, bais=False)
         # causal mask: token i can only attend to tokens <= i
        self.resiter_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
